# 04.2 Parameter Fitting: Exercises

## Contents

**Tools:** CADET-Process (Python).

**Python used:** Object creation, method calls, keyword arguments, lists, a dictionary and a simple function.

**Practises:** experimental references, `Comparator`, transport fitting, column characterization with an accepted system volume and comparing binding experiments.

1. Compare tracer data
2. Fit transport parameters
3. Characterize a newly packed column
4. Compare binding experiments (optional)

```{tip}
Run code cells from top to bottom with **Shift+Enter**. After entering, pasting or changing code, run that cell again to apply it, then rerun any later cells that use its results. Setup cells may show no output; they create objects for later cells. Fill empty answer cells as you go.
```

Extend the tracer fit to a second experiment and two detector positions, characterize a newly packed column on an instrument, then compare binding experiments with the initial binding estimates. Exercises marked **optional** extend the main tasks and can be skipped. The process models are supplied; the exercises connect them to the measurements. The workbooks are in `experimental_data`. Their time index is in **seconds**; concentration columns are in **mM** (numerically equal to mol/m³). The conductivity column is a raw signal in **mS/cm**.

## Exercise 1: compare tracer measurements

The supplied tracer process represents the detector positions in the flow sheet:

**Inlet → pipe5 → column → pipe6 (UV) → pipe7 (conductivity) → Outlet**

| Setting | Value |
|---|---|
| Tracer injection | Acetone, 1 mM for 1 s; about 1 mL/min |
| Column | Length 0.014 m; diameter 0.01 m; initial porosity 0.5; dispersion `5.75e-7` m²/s |
| Tubing lengths | `pipe5`: 0.1 m; `pipe6`: 0.02 m; `pipe7`: 0.1 m |
| All tubing | Diameter 0.001 m; dispersion `6e-6` m²/s; 50 axial cells |
| Observation time | 180 s |

The [setup script](../resources/parameter_fitting_exercises_setup.py) contains the complete model construction.

```{tip}
Run the cell below once with **Shift+Enter** to load the tracer process and simulate it with its initial estimates.
```

In [ ]:
%matplotlib inline
import pandas as pd
from CADETProcess.simulator import Cadet
# Load the supplied reference processes for the exercises.
%run ../resources/parameter_fitting_exercises_setup.py

process = tracer_process()
simulator = Cadet()
simulator.time_resolution = 0.5
simulation_results = simulator.simulate(process)

The next cell loads the measurements. `tracer_1.xlsx` contains the UV signal; `tracer_2.xlsx` contains a second UV signal and a conductivity signal. The conductivity is subtracted from its **0.1 mS/cm** baseline to obtain a positive response. This reverses the signal; it does not calibrate it to concentration.

In [ ]:
# Load both tracer experiments; time is the index, in seconds.
data1 = pd.read_excel("experimental_data/tracer_1.xlsx", index_col=0)
data2 = pd.read_excel("experimental_data/tracer_2.xlsx", index_col=0)

uv_1 = data1.iloc[:, 0]  # first column: UV signal
uv_2 = data2.iloc[:, 0]  # first column: UV signal; second column: conductivity
conductivity_2 = 0.1 - data2.iloc[:, 1]  # Reverse the signal around its baseline.

Create a `ReferenceIO` for each signal: `reference_uv1`, `reference_uv2` and `reference_conductivity`. Then create a `Comparator` named `comparator` and add one `Shape` metric per reference: the UV references at `"pipe6.outlet"`, conductivity at `"pipe7.outlet"`.

`Shape` compares alignment and peak shape, so it also works for the raw conductivity signal. An amplitude metric such as `SSE` would require calibrating conductivity to concentration first.

```{tip}
**Methods for this exercise**

- `ReferenceIO(name, time, signal)` stores a measured curve; use `data1.index` as time. See [`ReferenceIO`](https://cadet-process.readthedocs.io/en/latest/reference/generated/CADETProcess.reference.ReferenceIO.html).
- `Comparator()` collects difference metrics. See [`Comparator`](https://cadet-process.readthedocs.io/en/latest/reference/generated/CADETProcess.comparison.Comparator.html).
- `comparator.add_difference_metric(Shape(reference), "pipe6.outlet")` compares a reference with the simulated signal at that unit. See [`Shape`](https://cadet-process.readthedocs.io/en/latest/reference/generated/CADETProcess.comparison.Shape.html) and the [comparison guide](https://cadet-process.readthedocs.io/en/latest/user_guide/process_evaluation/comparison.html).
```

In [ ]:
# References and comparator
from CADETProcess.reference import ReferenceIO
from CADETProcess.comparison import Comparator, Shape

reference_uv1 = ReferenceIO("tracer experiment uv1", data1.index, uv_1)
reference_uv2 = ReferenceIO("tracer experiment uv2", data2.index, uv_2)
reference_conductivity = ReferenceIO("tracer experiment conductivity", data2.index, conductivity_2)

comparator = Comparator()
comparator.add_difference_metric(Shape(reference_uv1), "pipe6.outlet")
comparator.add_difference_metric(Shape(reference_uv2), "pipe6.outlet")
comparator.add_difference_metric(Shape(reference_conductivity), "pipe7.outlet")

The supplied cell evaluates the metrics and overlays each reference with its simulated signal.

In [ ]:
# Score and plot the initial model against all three signals.
print(comparator.evaluate(simulation_results))
_ = comparator.plot_comparison(simulation_results)

```{dropdown} Interpretation
Each reference is compared with the signal at its own detector position: the UV signals at `pipe6`, the conductivity signal further downstream at `pipe7`. `Shape` compares timing and peak shape, so the uncalibrated conductivity signal can be used. A shift between simulation and reference mainly points to porosity; a wider or narrower peak mainly points to dispersion.
```

## Exercise 2: fit transport parameters

Create an `OptimizationProblem` named `optimization_problem` and register the tracer `process`. Add the two variables **in this order**, keeping the tubing settings fixed:

| Variable name | Parameter path | Lower bound | Upper bound |
|---|---|---:|---:|
| `axial_dispersion` | `flow_sheet.column.axial_dispersion` | `1e-8` m²/s | `1e-5` m²/s |
| `porosity` | `flow_sheet.column.total_porosity` | 0.2 | 0.9 |

Use `normalization="auto"` for both, since their scales differ. Register the simulator as an evaluator, then add the comparator as the objective, passing the simulation results to it.

```{tip}
**Methods for this exercise** (as in the lesson)

- `OptimizationProblem(name)` and `optimization_problem.add_evaluation_object(process)`. See the [optimization problem guide](https://cadet-process.readthedocs.io/en/latest/user_guide/optimization/optimization_problem.html).
- `add_variable(name, parameter_path=..., lb=..., ub=..., normalization="auto")`.
- `add_evaluator(simulator)` and `add_objective(comparator, n_objectives=comparator.n_metrics, requires=[simulator])`. See the [evaluator guide](https://cadet-process.readthedocs.io/en/latest/user_guide/optimization/evaluator.html).
```

In [ ]:
# Optimization problem for the column transport
from CADETProcess.optimization import OptimizationProblem

optimization_problem = OptimizationProblem("column_transport")
optimization_problem.add_evaluation_object(process)

optimization_problem.add_variable(
    "axial_dispersion", parameter_path="flow_sheet.column.axial_dispersion",
    lb=1e-8, ub=1e-5, normalization="auto",
)
optimization_problem.add_variable(
    "porosity", parameter_path="flow_sheet.column.total_porosity",
    lb=0.2, ub=0.9, normalization="auto",
)

optimization_problem.add_evaluator(simulator)
optimization_problem.add_objective(
    comparator, n_objectives=comparator.n_metrics, requires=[simulator],
)

Check one candidate before the search: the initial estimates `[axial dispersion, porosity]` should return one score per metric.

In [ ]:
# Evaluate the initial estimates through the complete problem definition.
print(optimization_problem.evaluate_objectives([5.75e-7, 0.5]))

**Optional:** the search uses `U_NSGA3` with **64 candidates**, at most **12 generations** and **4 cores**. It takes several minutes; uncomment the `optimize()` call to run it, and use `n_cores = 1` on a smaller machine.

In [ ]:
# Optional search: uncomment the last lines to run it.
from CADETProcess.optimization import U_NSGA3

optimizer = U_NSGA3()
optimizer.n_cores = 4
optimizer.pop_size = 64
optimizer.n_max_iter = 12
optimizer.seed = 42

# optimization_results = optimizer.optimize(optimization_problem, use_checkpoint=False)
# print(optimization_results.x)

After fitting, compare each signal with its own reference. Which differences remain, and could the fixed instrument settings explain them?

```{dropdown} Interpretation
One set of column parameters must describe all three signals. Differences that remain in every signal suggest a model limitation; differences that appear only in one signal point to that detector position or to the fixed tubing settings. Parameters at a bound also call for checking the model and data before accepting the fit.
```

## Exercise 3: characterize a newly packed column

A column has just been packed (**10 cm × 2 cm**) and will be used for binding studies. Before that, its tracer-accessible **porosity** is needed. On the instrument, the detector does not see the column alone: the tracer also passes the **mixer and tubing**. A typical characterization therefore uses two tracer runs at the same flow rate (**6 mL/min**):

| Lab run | Setup | Measures |
|---|---|---|
| System run | 5 s tracer pulse, column replaced by a zero-volume union | Delay of mixer and tubing: the **system volume** |
| Column run | Continuous tracer feed, column connected | Delay of system and column together |

The system run was evaluated in **02.4 Residence Time Distributions**: the accepted system volume is **3 mL**. The data here are the synthetic measurements from that topic.

The supplied cell loads both runs, calculates their mean delays and stores the accepted system volume, so it can be reused in the column fit.

In [ ]:
# Load both lab runs and store the accepted system result.
import numpy as np
%run ../resources/characterization_data.py
pulse_data, step_data = load_rtd_measurements()
flow_rate = 1e-7  # m³/s (6 mL/min)
measured_system_delay = pulse_delay(pulse_data)
measured_total_delay = breakthrough_delay(step_data)
print(f"System run delay: {measured_system_delay:.1f} s")
print(f"Column run delay (system + column): {measured_total_delay:.1f} s")

accepted_parameters = {"system_volume": 3e-6}  # m³
column_volume = np.pi * 0.02 ** 2 * 0.1 / 4  # m³

**Step 1:** determine the porosity from the column run, with the accepted system volume held fixed. The predicted delay of the column run is

$$
\tau_\mathrm{total}=\frac{V_\mathrm{system}+\varepsilon_t V_\mathrm{column}}{Q}.
$$

Create an `OptimizationProblem` named `packing_problem` with one variable `porosity` between **0.2 and 0.6**. Write a function `packing_error(x)` that returns the squared difference between the predicted and the measured total delay, using `accepted_parameters["system_volume"]` and `x[0]` as porosity, and add it as the objective.

```{tip}
**Methods for this exercise**

- `OptimizationProblem(name)`, then `add_variable(name, lb=..., ub=...)`. Without an evaluation object, the variable is only a number passed to the objective. See the [optimization problem guide](https://cadet-process.readthedocs.io/en/latest/user_guide/optimization/optimization_problem.html).
- `add_objective(function, name=...)` registers a plain Python function that receives the candidate `x` and returns its score, as in [99.4 Optimization Basics](../../99%20Misc/99.4%20Optimization%20Basics/).
```

In [ ]:
# Porosity variable and delay objective
from CADETProcess.optimization import OptimizationProblem

packing_problem = OptimizationProblem("column_packing")
packing_problem.add_variable("porosity", lb=0.2, ub=0.6)

def packing_error(x):
    predicted_delay = (
        accepted_parameters["system_volume"] + x[0] * column_volume
    ) / flow_rate
    return (predicted_delay - measured_total_delay) ** 2

packing_problem.add_objective(packing_error, name="packing_error")

The supplied cell runs the fit, stores the accepted porosity and compares it with the value obtained by assigning the whole delay of the column run to the column, as if the system run had been skipped.

In [ ]:
# Fit, accept the porosity and compare with the column-only estimate.
from CADETProcess.optimization import NelderMead

packing_optimizer = NelderMead()
packing_optimizer.n_max_iter = 100
packing_results = packing_optimizer.optimize(packing_problem, x0=[0.4])
accepted_parameters["porosity"] = float(packing_results.x[0, 0])

apparent_porosity = measured_total_delay * flow_rate / column_volume
print(f"Ignoring the system: {apparent_porosity:.3f}")
print(f"With the accepted system volume: {accepted_parameters['porosity']:.3f}")
_ = plot_porosity_transfer(apparent_porosity, accepted_parameters["porosity"])

Why is the porosity smaller when the system volume is held fixed? Is the value of about 0.45 wrong, even though it lies between 0 and 1?

```{dropdown} Answer
The total delay contains about 30 s from the mixer and tubing. With the system volume fixed, only the remaining 110 s are assigned to the column, which gives a porosity near 0.35. The value of 0.45 is a possible volume fraction, but it describes column and instrument together. Carried into a model that contains mixer and tubing, as the binding studies will, the instrument delay would be counted twice and every retention time would be predicted too late.
```

**Step 2:** the column run also shows how much the front spreads, i.e. the column dispersion. Can the delay fit determine it as well? The supplied cell simulates the instrument with the accepted values and two dispersions, `1e-7` and `1e-6` m²/s, and compares them with the measured column run. Compare mean delay and curve RMSE in the table.

In [ ]:
# Compare two dispersions at the accepted porosity and system volume.
curve_checks = check_characterized_column(accepted_parameters, step_data)

```{dropdown} Answer
Both dispersions give almost the same mean delay, so the delay objective cannot tell them apart. The larger dispersion spreads the breakthrough front and has a clearly larger RMSE. Dispersion therefore needs a full-curve metric, as in Exercise 2; the reference was generated with `1e-7` m²/s.
```

The accepted system volume, porosity and dispersion now describe the transport of this setup. They stay fixed in the binding studies, as in the optional Exercise 4.

## Exercise 4 (optional): compare binding experiments

Three experiments constrain the binding parameters: a breakthrough and two salt gradients of **15 and 120 column volumes (CV)**. Each workbook contains **Salt, A, B and C** in that order. Column and tubing transport stay fixed while examining binding.

The supplied processes use the column and tubing from Exercise 1, with components **Salt, A, B, C** and initial `StericMassAction` estimates:

| Setting | Salt, A, B, C |
|---|---|
| `adsorption_rate` | `[0, 2.3e-5, 0.559, 3.5e-3]` |
| `desorption_rate` | `[0, 1, 1, 1]` |
| `characteristic_charge` | `[1, 6.9, 2.3, 5.8]` (salt valence first) |
| `steric_factor` | `[0, 10, 10.6, 11.83]` |
| `capacity` | `1200` (single value) |

| Experiment | Method |
|---|---|
| Breakthrough | Feed **50 mM salt + 1 mM each of A, B, C** from 0 s; cycle time **60 min** |
| 15 and 120 CV gradients | Load **50 mM salt + 1 mM each protein** for **10 s**; wash at **50 mM salt** until **90 s**; then increase salt linearly from **80 to 500 mM** over the gradient length, counting **60 s per CV**; then hold **500 mM**. Cycle time is the gradient duration plus **600 s** |

```{tip}
Run the cell below once with **Shift+Enter** to create the three processes, load their measurements and simulate the initial estimates. The 120 CV gradient takes a little longer.
```

In [ ]:
from CADETProcess.processModel import ComponentSystem

component_system = ComponentSystem(["Salt", "A", "B", "C"])
process_bt = breakthrough_process(component_system)
process_15cv = gradient_process(15, component_system)
process_120cv = gradient_process(120, component_system)

data_bt = pd.read_excel("experimental_data/breakthrough.xlsx", index_col=0)
data_15cv = pd.read_excel("experimental_data/15cv_gradient.xlsx", index_col=0)
data_120cv = pd.read_excel("experimental_data/120cv_gradient.xlsx", index_col=0)

results_bt = simulator.simulate(process_bt)
results_15cv = simulator.simulate(process_15cv)
results_120cv = simulator.simulate(process_120cv)

For each experiment, create a `ReferenceIO` from its complete workbook (`reference_bt`, `reference_15cv`, `reference_120cv`). Then create one `Comparator` per experiment (`comparator_bt`, `comparator_15cv`, `comparator_120cv`) with a `Shape` metric for each protein **A, B and C** at `"pipe6.outlet"`.

```{tip}
**Methods for this exercise**

- With several components, pass the component system: `ReferenceIO(name, data_bt.index, data_bt, component_system=component_system)`. See [`ReferenceIO`](https://cadet-process.readthedocs.io/en/latest/reference/generated/CADETProcess.reference.ReferenceIO.html).
- `Comparator(name=...)` gives each experiment its own comparator.
- `Shape(reference, components=["A"])` compares only protein A; add one metric per protein. See [`Shape`](https://cadet-process.readthedocs.io/en/latest/reference/generated/CADETProcess.comparison.Shape.html).
```

In [ ]:
# References and comparators per experiment
reference_bt = ReferenceIO("experiment bt", data_bt.index, data_bt, component_system=component_system)
reference_15cv = ReferenceIO("experiment 15cv", data_15cv.index, data_15cv, component_system=component_system)
reference_120cv = ReferenceIO("experiment 120cv", data_120cv.index, data_120cv, component_system=component_system)

comparator_bt = Comparator(name=reference_bt.name)
comparator_15cv = Comparator(name=reference_15cv.name)
comparator_120cv = Comparator(name=reference_120cv.name)
comparator_bt.add_difference_metric(Shape(reference_bt, components=["A"]), "pipe6.outlet")
comparator_bt.add_difference_metric(Shape(reference_bt, components=["B"]), "pipe6.outlet")
comparator_bt.add_difference_metric(Shape(reference_bt, components=["C"]), "pipe6.outlet")
comparator_15cv.add_difference_metric(Shape(reference_15cv, components=["A"]), "pipe6.outlet")
comparator_15cv.add_difference_metric(Shape(reference_15cv, components=["B"]), "pipe6.outlet")
comparator_15cv.add_difference_metric(Shape(reference_15cv, components=["C"]), "pipe6.outlet")
comparator_120cv.add_difference_metric(Shape(reference_120cv, components=["A"]), "pipe6.outlet")
comparator_120cv.add_difference_metric(Shape(reference_120cv, components=["B"]), "pipe6.outlet")
comparator_120cv.add_difference_metric(Shape(reference_120cv, components=["C"]), "pipe6.outlet")

In [ ]:
# Compare the initial binding estimates with each experiment.
_ = comparator_bt.plot_comparison(results_bt)
_ = comparator_15cv.plot_comparison(results_15cv)
_ = comparator_120cv.plot_comparison(results_120cv)

```{dropdown} Interpretation
The starting binding parameters are initial estimates, so the curves need not match yet. The experiments contain different information: the breakthrough mainly constrains capacity and steric shielding, while two gradients of different length help separate salt dependence (characteristic charge) from binding strength. Fitting all experiments together therefore constrains more parameters than any single experiment.
```

```{note}
**Fitting the binding parameters** follows the same pattern as the transport fit in Exercise 2: an `OptimizationProblem` with the processes as evaluation objects, variables pointing to the binding model's parameters, for example `flow_sheet.column.binding_model.characteristic_charge`, the simulator as evaluator and the comparators as objectives. Because one parameter set must describe all three experiments, each comparator scores only its own process. The complete fit is in **99.5 Advanced Optimization: Exercises**.
```